# 02 — Modelo Baseline para Regressão

Este notebook apresenta um template didático para criação de um modelo baseline de **regressão**.

Use este notebook quando o objetivo for prever um **valor numérico contínuo**, por exemplo:

- preço de imóvel;
- valor de venda;
- salário;
- temperatura;
- demanda;
- faturamento.

## Premissa

Este template pressupõe que o dataset já passou pela etapa de preparação dos dados.

Ou seja:

- não há valores nulos relevantes;
- as variáveis categóricas já foram tratadas;
- as variáveis preditoras estão prontas para o modelo;
- a variável-alvo já está definida.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## 1. Carregar o dataset

Ajuste o caminho do arquivo conforme o local onde o dataset estiver salvo.

Exemplos de datasets didáticos para regressão:

- California Housing Prices — alvo: `median_house_value`
- Housing Prices Dataset — alvo: preço do imóvel
- Student Performance — alvo: nota ou desempenho numérico

In [ ]:
df = pd.read_csv("dataset_regressao.csv")

df.head()

## 2. Visualização inicial dos dados

In [ ]:
print("Linhas e colunas:", df.shape)

df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.isnull().sum()

## 3. Visualização gráfica dos dados

In [ ]:
df.select_dtypes(include=np.number).hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

In [ ]:
df.select_dtypes(include=np.number).plot(
    kind="box",
    figsize=(12, 6),
    rot=45
)

plt.title("Boxplot das variáveis numéricas")
plt.show()

In [ ]:
correlacao = df.select_dtypes(include=np.number).corr()

plt.figure(figsize=(10, 6))
plt.imshow(correlacao, aspect="auto")
plt.colorbar()
plt.xticks(range(len(correlacao.columns)), correlacao.columns, rotation=90)
plt.yticks(range(len(correlacao.columns)), correlacao.columns)
plt.title("Matriz de correlação")
plt.show()

## 4. Separar variáveis preditoras e variável-alvo

In [ ]:
target = "alvo"  # Altere para o nome da coluna alvo

X = df.drop(columns=[target])
y = df[target]

print("Variáveis preditoras:")
print(X.head())

print("\nVariável-alvo:")
print(y.head())

## 5. Visualizar a distribuição da variável-alvo

Em regressão, é importante observar se o alvo tem distribuição muito assimétrica ou presença de outliers.

In [ ]:
plt.hist(y, bins=30)
plt.title("Distribuição da variável-alvo")
plt.xlabel(target)
plt.ylabel("Frequência")
plt.show()

## 6. Dividir treino e teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

## 7. Normalização

A normalização é importante para modelos sensíveis à escala, como regressão linear com regularização, KNN, SVM e redes neurais.

Para árvores de decisão, normalização normalmente não é obrigatória.

Regra importante:

- `fit_transform` apenas no treino;
- `transform` no teste.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Outras opções:
# scaler = MinMaxScaler()
# scaler = RobustScaler()

## 8. Baseline ingênuo

O `DummyRegressor` cria um modelo extremamente simples.

A estratégia `mean` sempre prevê a média do alvo no conjunto de treino.

O modelo real precisa ser melhor do que esse baseline.

In [ ]:
dummy_regressor = DummyRegressor(strategy="mean")

dummy_regressor.fit(X_train, y_train)

y_pred_dummy = dummy_regressor.predict(X_test)

## 9. Modelo baseline real

Para fins didáticos, podemos usar uma árvore de decisão para regressão.

Hiperparâmetros:

- `max_depth`: profundidade máxima da árvore;
- `min_samples_split`: mínimo de amostras para dividir um nó;
- `random_state`: garante reprodutibilidade.

In [ ]:
model = DecisionTreeRegressor(
    max_depth=4,
    min_samples_split=10,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

## 10. Avaliação do modelo de regressão

Métricas principais:

- **MAE**: erro absoluto médio.
- **MSE**: erro quadrático médio.
- **RMSE**: raiz do erro quadrático médio.
- **R²**: quanto da variação do alvo o modelo consegue explicar.
- **MAPE**: erro percentual absoluto médio.

Em regressão, normalmente não falamos em percentual de acerto.
Falamos em erro.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=== Modelo Baseline Real ===")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R²:", r2)

## 11. Comparar com baseline ingênuo

In [ ]:
mae_dummy = mean_absolute_error(y_test, y_pred_dummy)
mse_dummy = mean_squared_error(y_test, y_pred_dummy)
rmse_dummy = np.sqrt(mse_dummy)
r2_dummy = r2_score(y_test, y_pred_dummy)

print("=== Baseline Ingênuo ===")
print("MAE Dummy:", mae_dummy)
print("RMSE Dummy:", rmse_dummy)
print("R² Dummy:", r2_dummy)

print("\n=== Modelo Real ===")
print("MAE Modelo:", mae)
print("RMSE Modelo:", rmse)
print("R² Modelo:", r2)

## 12. MAPE

O MAPE mede o erro percentual médio.

Atenção: o MAPE pode gerar problemas quando `y_test` possui valores iguais ou muito próximos de zero.

In [ ]:
y_test_array = np.array(y_test)
y_pred_array = np.array(y_pred)

mask = y_test_array != 0

if mask.sum() > 0:
    mape = np.mean(np.abs((y_test_array[mask] - y_pred_array[mask]) / y_test_array[mask])) * 100
    print(f"MAPE: {mape:.2f}%")
else:
    print("MAPE não calculado porque todos os valores reais são zero.")

## 13. Visualizar previsto versus real

Quanto mais os pontos se aproximarem de uma linha diagonal imaginária, melhor a relação entre valor real e valor previsto.

In [ ]:
plt.scatter(y_test, y_pred)
plt.xlabel("Valor real")
plt.ylabel("Valor previsto")
plt.title("Valores reais x valores previstos")
plt.show()

## 14. Predição com uma amostra real do teste

In [ ]:
vetor_teste = X_test.iloc[[0]]

predicao = model.predict(vetor_teste)
valor_real = y_test.iloc[0]

erro_absoluto = abs(predicao[0] - valor_real)

print("Vetor de teste:")
print(vetor_teste)

print("\nPredição:", predicao[0])
print("Valor real:", valor_real)
print("Erro absoluto:", erro_absoluto)

if valor_real != 0:
    erro_percentual = erro_absoluto / abs(valor_real) * 100
    print(f"Erro percentual da amostra: {erro_percentual:.2f}%")

## 15. Predição com vetor manual

Aqui criamos um registro simples usando mediana para variáveis numéricas e moda para variáveis categóricas.

In [ ]:
novo_registro = pd.DataFrame([{
    col: X[col].median() if np.issubdtype(X[col].dtype, np.number) else X[col].mode()[0]
    for col in X.columns
}])

predicao_novo = model.predict(novo_registro)

print("Novo registro:")
print(novo_registro)

print("\nValor previsto:", predicao_novo[0])

## Sugestões didáticas

1. Explique que regressão prevê número contínuo.
2. Evite falar em acerto exato; fale em erro.
3. Compare sempre com `DummyRegressor`.
4. Use MAE para interpretação simples.
5. Use RMSE quando quiser penalizar erros grandes.
6. Use R² para explicar poder explicativo do modelo.